In [21]:
import os
from utils_extraction import extract_body, tokenize, clean_tokens, decode
from utils_extraction import chunk_tokens, flatten_token_chunks
from utils_extraction import extract_few_shot_examples
from utils_extraction import select_few_shot 
from utils_extraction import merge_tokens_with_auto_labels, add_attributes_to_auto_labels, compare_html_allow_auto_labels, correct_tokens_brackets, check_tokens_brackets
from models import GPTAssistant
from utils_extraction import process_chunks
from utils_extraction.html_utils import clean_html_formatting

In [60]:
# ---------- Define Hyperparameters ----------
min_tokens = 500
fs_min_tokens = 100
model_name = "o1"

n_few_shot = 30  # Number of few-shot examples to use

#### Define the text to process, and where to save it. Define the text for few shot

In [61]:
# File paths
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
filename = "1989CanLII1415ONCA" #"1989CanLII1415ONCA" #"2021QCCA1675" #"1997CanLII16226_ONCA"
round = "ronde_2"
anno = "llm"
version = "v1.0"
html_path = fr"{project_root}\data\Document_Échantillon_Initial\{round}\plain_html_arbre_balise\{filename}.html"
output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}\test"

os.makedirs(output_dir, exist_ok=True)
# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")


fs_filename = "2019SCC65_annotated_EG_v1_corrected"
fs_anno = "EG"
fs_version = "v1"
fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}.html"
# Read HTML file
with open(fs_html_path, 'r', encoding='utf-8') as file:
    fs_html_content = file.read()
print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")


   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Document_Échantillon_Initial\ronde_2\plain_html_arbre_balise\1989CanLII1415ONCA.html
   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\EG\2019SCC65_annotated_EG_v1_corrected.html


### Process The HTML Content

In [62]:

# ---------- Extract body content ----------
body_content = extract_body(html_content)


# ---------- Tokenize body content ----------
tokens = tokenize(body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
#token_chunks = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)


In [45]:
for tok1, tok2 in zip(normalized_cleaned_tokens, tokenize(decode(normalized_cleaned_tokens))):
    if tok1 != tok2:
        print(tok1, tok2)
        break

In [26]:
import spacy
import re

nlp = spacy.load("en_core_web_trf")

doc = nlp(decode(normalized_cleaned_tokens))
initial_sentences = [sent.text for sent in doc.sents]



In [46]:
initial_sentences_token = [tokenize(sent) for sent in initial_sentences]
flat_initial_sentences = flatten_token_chunks(initial_sentences_token, separator="<sep>")

   ✓ Flattened 236 chunks into 9642 tokens


In [47]:
def merge_tokens_general(original_tokens: list[str], 
                        derived_tokens: list[str], 
                        is_protected_func,
                        log: bool = False) -> list[str]:
    """
    GENERALIZED VERSION: Merge original tokens with derived tokens.
    
    Goal: Produce the original text with protected tokens (e.g., <sep>, <auto_label>) 
    inserted from the derived version.
    
    Algorithm:
    - If tokens are equivalent (same or both whitespace): take original token, advance both indices
    - If tokens differ:
      - If derived token is protected: it's an insertion, take it and advance idx2 only
      - Otherwise: try to merge consecutive original tokens to match derived token
      - If no merge possible: take original token and advance idx1 only
    
    This assumes derived is mostly a superset of original (original + protected tokens).
    
    Args:
        original_tokens: Original token list (without protected tokens)
        derived_tokens: Derived token list (with protected tokens inserted)
        is_protected_func: Function that takes a token and returns True if it's protected
        log: Print debug information
    
    Returns:
        Merged token list with original tokens + protected tokens from derived
    
    Example:
        original = ['Act', ',', 'section', '5']
        derived = ['Act', '<sep>', ',', 'section', '<sep>', '5']
        is_protected = lambda t: t == '<sep>'
        -> ['Act', '<sep>', ',', 'section', '<sep>', '5']
    """
    n1 = len(original_tokens)
    n2 = len(derived_tokens)
    result = []
    idx1 = 0
    idx2 = 0
    
    def tokens_equivalent(tok1: str, tok2: str) -> bool:
        """Check if two tokens are equivalent (exact match or both whitespace)."""
        if tok1 == tok2:
            return True
        # Both are pure whitespace
        if not tok1.strip() and not tok2.strip():
            return True
        return False
    
    def try_merge_original_to_match_derived(start_idx: int, target: str) -> int:
        """
        Try to merge consecutive original tokens to match the derived token.
        
        Example: If original=['générale', '"'] and derived='généraleˮ',
        this will detect that original[0] + original[1] matches derived.
        
        Args:
            start_idx: Starting index in original_tokens
            target: The derived token to match
            
        Returns:
            Number of original tokens that combine to match target (0 if no match)
        """
        if start_idx >= n1:
            return 0
        
        accumulated = ""
        # Look ahead up to 10 tokens to find a match
        for i in range(start_idx, min(start_idx + 10, n1)):
            accumulated += original_tokens[i]
            if accumulated == target:
                return i - start_idx + 1
        return 0
    
    while idx1 < n1 and idx2 < n2:
        t1 = original_tokens[idx1]
        t2 = derived_tokens[idx2]
        
        if tokens_equivalent(t1, t2):
            # Tokens match: keep original and advance both
            result.append(t1)
            if log:
                print(f"Match: '{t1}' == '{t2}' -> '{t1}'")
            idx1 += 1
            idx2 += 1
        else:
            # Tokens differ
            if is_protected_func(t2):
                # t2 is a protected token (insertion in derived version)
                result.append(t2)
                if log:
                    print(f"Protected: '{t1}' vs '{t2}' -> '{t2}'")
                idx2 += 1
            else:
                # Neither matches nor is protected
                # Try to merge original tokens to match derived token
                merge_count = try_merge_original_to_match_derived(idx1, t2)
                
                if merge_count > 0:
                    # Found a match by merging multiple original tokens
                    for i in range(merge_count):
                        result.append(original_tokens[idx1 + i])
                    if log:
                        merged_tokens = original_tokens[idx1:idx1+merge_count]
                        print(f"Merged {merge_count} tokens: {merged_tokens} -> '{t2}'")
                    idx1 += merge_count
                    idx2 += 1
                else:
                    # No merge possible: keep original token
                    result.append(t1)
                    if log:
                        print(f"Diff: '{t1}' vs '{t2}' -> '{t1}'")
                    idx1 += 1
    
    # Append remaining tokens from original (if any)
    if idx1 < n1:
        result.extend(original_tokens[idx1:])
    
    # Append remaining tokens from derived (if any, likely protected tokens)
    if idx2 < n2:
        result.extend(derived_tokens[idx2:])
    
    if log:
        print(f"   ✓ Merged {n1} original + {n2} derived → {len(result)} tokens")
        print(f"   ✓ Added {len(result) - n1} protected tokens")
    
    return result

In [48]:
is_sep_tag = lambda token: token == '<sep>'

# Example: merge normalized_cleaned_tokens (original, no <sep>) 
# with flat_token_sentence_chunks (derived, with <sep>)

print(f"Original tokens: {len(normalized_cleaned_tokens)} (no <sep>)")
print(f"Derived tokens: {len(flat_initial_sentences)} (with <sep>)")

# This assumes flat_token_sentence_chunks is normalized_cleaned_tokens + <sep> insertions
corrected_initial_sentences = merge_tokens_general(
    original_tokens=normalized_cleaned_tokens,
    derived_tokens=flat_initial_sentences,
    is_protected_func=is_sep_tag,
    log=False
)

print(f"\nResult: {len(corrected_initial_sentences)} tokens")
print(f"Number of <sep> tags: {corrected_initial_sentences.count('<sep>')}")

Original tokens: 9605 (no <sep>)
Derived tokens: 9642 (with <sep>)

Result: 9840 tokens
Number of <sep> tags: 235


In [49]:
def calculate_combined_density(sentence):
    """Calculate combined period and number density for a sentence."""
    if len(sentence) < 10:
        return 0
    
    num_periods = sentence.count('.')
    num_digits = sum(c.isdigit() for c in sentence)
    
    period_density = (num_periods / len(sentence)) * 100
    digit_density = (num_digits / len(sentence)) * 100
    
    return period_density + digit_density

def detect_citation_sections_sequential(sentences, threshold=25, consecutive_gap=3):
    """
    Detect citation sections using sequential analysis.
    
    Algorithm:
    1. Go through sentences one by one
    2. When density > threshold, start a citation section
    3. Continue until we find 'consecutive_gap' sentences below threshold
    4. End the citation section 'consecutive_gap' sentences before
    5. STOP - all remaining sentences are NOT citations
    
    Args:
        sentences: List of sentences
        threshold: Combined density threshold (%) to consider citation
        consecutive_gap: Number of consecutive non-citation sentences to end section
    
    Returns:
        List of booleans indicating if each sentence is in a citation section
    """
    is_citation = [False] * len(sentences)
    in_citation_section = False
    citation_start = None
    below_threshold_count = 0
    citation_section_ended = False  # Track if we've already found and ended a citation section
    
    for i, sent in enumerate(sentences):
        # If citation section has already ended, all remaining sentences are NOT citations
        if citation_section_ended:
            break
        
        density = calculate_combined_density(sent)
        
        if density > threshold:
            # This is a citation sentence
            if not in_citation_section:
                # Start new citation section
                in_citation_section = True
                citation_start = i
            # Reset the gap counter
            below_threshold_count = 0
            is_citation[i] = True
            
        else:
            # Below threshold
            if in_citation_section:
                # We're in a citation section, count consecutive non-citations
                below_threshold_count += 1
                
                if below_threshold_count >= consecutive_gap:
                    # End citation section: go back 'consecutive_gap' sentences
                    citation_end = i - consecutive_gap
                    # Mark all sentences in this section
                    for j in range(citation_start, citation_end + 1):
                        is_citation[j] = True
                    # STOP HERE - citation section has ended
                    citation_section_ended = True
                    in_citation_section = False
    
    # Handle case where document ends while in citation section
    if in_citation_section and citation_start is not None:
        citation_end = len(sentences) - 1 - below_threshold_count
        for j in range(citation_start, citation_end + 1):
            is_citation[j] = True
    
    return is_citation

def merge_sentences_with_heuristics_tokens(tokens, citation_threshold=25, min_token=500):
    """
    Merge sentences based on boundary heuristics, working with tokens.
    
    Input : Flat list of tokens with <sep> as sentence boundaries.
    Output: Flat list of tokens with selective <sep> removal based on citation detection.

    Rules:
    - Normal section: Keep <sep> only if sentence ends with '.'
    - Citation section: Keep <sep> only if sentence ends with ';'
    - Otherwise: Merge sentences together (remove <sep>)
    
    Args:
        tokens: Flat List of token containing <sep> as sentence boundaries
        citation_threshold: Combined density threshold (%) for citation detection

    Returns:
        Flat list of tokens with selective <sep> removal based on citation detection
    """
    if not tokens:
        return []
    
    # STEP 1: Split into sentences based on <sep>
    sentences_tokens = []
    current_sentence = []
    for token in tokens:
        if token == '<sep>':
            if current_sentence:
                sentences_tokens.append(current_sentence)
                current_sentence = []
        else:
            current_sentence.append(token)
    
    if current_sentence:
        sentences_tokens.append(current_sentence)

    # STEP 2: Convert to text for citation detection
    sentences_text = [decode(sent) for sent in sentences_tokens]

    # STEP 3: Detect citations section
    is_citation = detect_citation_sections_sequential(sentences_text, threshold=citation_threshold)

    # STEP 4: Reconstruct flat token list with selective <sep> removal
    result = []
    current_chunk_size = 0  # Track tokens since last <sep>
    
    for i, sent_tokens in enumerate(sentences_tokens):
        result.extend(sent_tokens)
        current_chunk_size += len(sent_tokens)

        # Decide wheter to add <sep> after this sentence
        if i < len(sentences_tokens) - 1:  # Not the last sentence
            should_keep_sep = False

            # Get the last token of current sentence
            last_token = sent_tokens[-1] if sent_tokens else ''

            # Only consider keeping <sep> if we have at least min_token accumulated
            if current_chunk_size >= min_token:
                if is_citation[i]:
                    # Citation section: keep <sep> only if last token is with ';'
                    if last_token == ";":
                        should_keep_sep = True
                else:
                    # Normal section: keep <sep> only if last token is with '.'
                    if last_token == ".":
                        should_keep_sep = True
            
            if should_keep_sep:
                result.append('<sep>')
                current_chunk_size = 0  # Reset chunk size after keeping <sep>
    
    return result

In [50]:
# Apply heuristic-based merging with sequential citation detection
CITATION_THRESHOLD = 25  # Combined density threshold (%) - adjust based on the graphs above

flat_token_sentence_chunks = merge_sentences_with_heuristics_tokens(corrected_initial_sentences, citation_threshold=CITATION_THRESHOLD)
print(f"Initial sentences: {len(corrected_initial_sentences)}, After merging: {len(flat_token_sentence_chunks)}")
print(f"Using citation threshold: {CITATION_THRESHOLD}% (combined period + number density)")
print(f"Gap tolerance: 3 consecutive sentences below threshold to end citation section")
print(f"Note: Only the FIRST citation section is detected; all subsequent sentences are not citations")

Initial sentences: 9840, After merging: 9622
Using citation threshold: 25% (combined period + number density)
Gap tolerance: 3 consecutive sentences below threshold to end citation section
Note: Only the FIRST citation section is detected; all subsequent sentences are not citations


In [51]:
token_chunks =  []
current_chunk = []

for token in flat_token_sentence_chunks:

    if token != "<sep>":
        current_chunk.append(token)
    else:
        token_chunks.append(current_chunk)
        current_chunk =  []
token_chunks.append(current_chunk)

In [52]:
# Verify: merged should equal normalized_cleaned_tokens if ignoring <sep> tag

if flatten_token_chunks(token_chunks) == normalized_cleaned_tokens:
    print("✓ Perfect match! Derived was indeed original + <sep> insertions")
else:
    print("⚠ Some differences exist beyond <sep> insertions")
    # Show first difference
    for i, (m, d) in enumerate(zip(flat_token_sentence_chunks, flat_token_sentence_chunks)):
        if m != d:
            print(f"  First diff at index {i}: merged='{m}' vs derived='{d}'")
            break
    assert "Difference detected"

   ✓ Flattened 18 chunks into 9605 tokens
✓ Perfect match! Derived was indeed original + <sep> insertions


In [53]:
# ---------- Extract body content ----------
fs_body_content = extract_body(fs_html_content)


# ---------- Tokenize body content ----------
fs_tokens = tokenize(fs_body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=fs_tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunk1, token_chunk2 = chunk_tokens(normalized_cleaned_tokens, min_tokens=fs_min_tokens, stop_bookmark_separation=True)

   ✓ Found bookmark separator at index 43561
   ✓ Splitting: 43561 tokens before, 102341 tokens after
   ✓ Chunked tokens into 278 chunks (>= 100 tokens each)
   ✓ Chunked tokens into 1000 chunks (>= 100 tokens each)
   ✓ Total chunks: 278 before + 1000 after = 1278


In [54]:
label_config = {
    "keep_attributes":["labelname"], # extraction only, no disambiguation
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    "keep_labels":["decision", "legislation", "secondary sources"]
}

In [55]:
# ---------- Create few-shot examples ----------

few_shot_examples = extract_few_shot_examples(token_chunk1, 
                                              label_config)



selected_few_shot_examples = select_few_shot(examples=few_shot_examples, n=n_few_shot, 
                                             method="distributed", 
                                             list_of_labels=["decision", "legislation", "secondary sources"], 
                                             distribution=[0.3, 0.3, 0.3])
print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")

   ✓ Extracted 278 few-shot examples from chunks
   ⚠ Warning: Requested 9 examples with label 'secondary sources', but only 5 available
   ✓ Selected 30 few-shot examples for processing.


In [56]:
for examples in selected_few_shot_examples:
    print("==============================================================================")
    print("input: \n ", examples[0])
    print("------------------------------------------------------------------------------")
    print("output: \n ", examples[1])

input: 
   the Registrar’s decision is reasonableness. The Registrar’s decision was unreasonable. She failed to justify her interpretation of s. 3(2)(a) in light of the constraints imposed by s. 3 considered as a whole, by international treaties that inform its purpose
------------------------------------------------------------------------------
output: 
   the Registrar’s decision is reasonableness. The Registrar’s decision was unreasonable. She failed to justify her interpretation of <legislation>s. 3(2)(a)</legislation> in light of the constraints imposed by <legislation>s. 3</legislation> considered as a whole, by international treaties that inform its purpose
input: 
   promise of simplicity and predictability in this respect has not been fully realized. In Dunsmuir, a majority of the Court merged the standards of “patent unreasonableness” and “reasonableness simpliciter” into a single “reasonableness” standard, thus reducing the number of standards of review from three
---------

In [63]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name, temperature=1)

In [64]:
from utils_extraction import prepare_label_tokens, apply_post_processing_transforms, distance_lists_auto_label, apply_operations_safe, verify_processed_chunk
from utils_extraction import get_prompt_processing
import json 
from tqdm import tqdm 

class ProcessingHistory:
    """Track processing history for debugging and analysis."""
    
    def __init__(self):
        self.entries = []
    
    def add(self, status: str, chunk_idx: int, raw_output: str, error_details: str = None):
        """Add an entry to the history."""
        self.entries.append({
            "status": status,
            "chunk_idx": chunk_idx,
            "raw_output": raw_output,
            "error_details": error_details
        })
    
    def save(self, output_dir: str, filename: str):
        """Save history to JSON file."""
        if not output_dir or not filename:
            return
        
        json_path = os.path.join(output_dir, f"history_{filename}.json")
        try:
            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(self.entries, f, ensure_ascii=False, indent=4)
            print(f"   ✓ Processing history saved to: {json_path}")
        except Exception as e:
            print(f"   ✗ Error saving history: {e}")
    
    def summary(self) -> dict:
        """Get summary statistics."""
        statuses = [entry["status"] for entry in self.entries]
        return {
            "total": len(statuses),
            "success": statuses.count("Success"),
            "hallucination_fail": statuses.count("Hallucination Fail"),
            "consistency_fail": statuses.count("Consistency Fail"),
            "label_scheme_fail": statuses.count("Label Scheme Fail"),
            "double_hallucination_fail": statuses.count("Double Hallucination Fail"),
            "double_consistency_fail": statuses.count("Double Consistency Fail")
        }


def process_single_chunk(
    model,
    chunk: list,
    system_prompt: str,
    user_prompt_template: str,
    label_config: dict,
    allowed_labels = None,
) :
    """
    Process a single chunk with post-processing, verification, and fallback.
    
    Pipeline:
    1. Prepare input (clean chunk)
    2. Generate LLM output
    3. Post-process output (extract, transform)
    4. Verify output (hallucination, consistency, label scheme)
    
    Args:
        model: LLM model instance
        chunk: List of tokens to process
        system_prompt: System prompt for LLM
        user_prompt_template: User prompt template with {text} placeholder
        label_config: Configuration for label transformations
        allowed_labels: List of allowed label names for scheme validation
        max_fallback_attempts: Maximum number of fallback attempts per error type
    
    Returns:
        Tuple of (processed_tokens, status, error_details)
        - processed_tokens: Successfully processed tokens or original chunk if failed
        - status: "Success", "Hallucination Fail", "Consistency Fail", etc.
        - error_details: Description of error if failed, None if success
    """    
    # ------ 1. PREPARE INPUT ------
    cleaned_chunk = prepare_label_tokens(
        chunk,
        label_config={
            "keep_attributes": ["labelname"],
            "switch_type": False,
            "use_simplified": False
        }
    )
    text = decode(cleaned_chunk)
    
    # ------ 2. GENERATE LLM OUTPUT ------
    user_prompt = user_prompt_template.format(text=text)
    raw_output = model.generate(
        system_prompt=system_prompt,
        user_prompt=user_prompt
    )

    saving_raw_output = tokenize(raw_output)  # Keep a copy for history/logging before any processing

    #print(f"   → Raw LLM output for chunk: {raw_output}...")  
    
    # ------ 3. POST-PROCESS OUTPUT ------
    try:
        processed_tokens = apply_post_processing_transforms(
            raw_output=raw_output,
            use_simplified=label_config.get("use_simplified", False),
            label_type='auto_label'
        )
    except Exception as e:
        return chunk, "Post-processing Error", f"Failed to post-process: {str(e)}"
    
    #print(f"   → Processed tokens before verification: {decode(processed_tokens)}")
    
    # ------ 4. APPLY ERROR CORRECTION (BEFORE VERIFICATION) ------
    # This aligns tokens to handle minor discrepancies
    _, operations = distance_lists_auto_label(cleaned_chunk, processed_tokens)
    processed_tokens_corrected = apply_operations_safe(processed_tokens, operations)
    
    # ------ 5. VERIFY OUTPUT ------
    saving_raw_output_t = []
    for el in saving_raw_output:
        if not el.startswith("<"):
            saving_raw_output_t.append(el)

    test_verification = verify_processed_chunk(
        original_tokens=cleaned_chunk,
        processed_tokens=saving_raw_output_t,
        allowed_labels=allowed_labels,
        check_scheme=True
    )

    #print(f"Processed tokens after correction: {decode(processed_tokens_corrected)}")
    verification = verify_processed_chunk(
        original_tokens=cleaned_chunk,
        processed_tokens=processed_tokens_corrected,
        allowed_labels=allowed_labels,
        check_scheme=True
    )

    if not test_verification.passed :
        print(cleaned_chunk)
        print(saving_raw_output)

        for ori, de in zip(cleaned_chunk, saving_raw_output_t):
            if ori != de:
                print(f"Original: '{ori}' vs Derived: '{de}'")
        
    
    if verification.passed:
        return processed_tokens_corrected, "Success", None
    
    # ------ 6. HANDLE VERIFICATION FAILURES ------
    print(f"   ⚠ Verification failed: {verification.error_type}")
    print(f"   Details: {verification.details}")
    
    if verification.error_type == "hallucination":
        failure_type = "Hallucination Fail"
    elif verification.error_type == "consistency":
        failure_type = "Consistency Fail"
    elif verification.error_type == "label_scheme":
        failure_type = "Label Scheme Fail"
    
    
    return chunk, failure_type, verification.details




# ============================================================
# =========== MAIN FUNCTION FOR CHUNCK PROCESSING  ===========
# ============================================================


def process_chunks(
    model,
    token_chunks: list,
    process_prompt_path: str,
    label_config: dict,
    few_shot_examples = None,
    allowed_labels = None,
    output_dir= None,
    filename = None,
) -> list:
    """
    Process multiple chunks using AI model with verification and fallback.
    
    This is the main entry point for chunk processing. It coordinates:
    - LLM generation
    - Post-processing transformations
    - Verification checks
    - Error correction and fallbacks
    - History tracking
    
    Args:
        model: LLM model instance
        token_chunks: List of token chunks to process
        process_prompt_path: Path to the main processing prompt file
        label_config: Configuration for label transformations
        few_shot_examples: Optional list of (input, output) examples
        allowed_labels: Optional list of allowed label names
        output_dir: Optional directory to save outputs
        filename: Optional filename prefix for outputs
        max_fallback_attempts: Maximum fallback attempts per error type
    
    Returns:
        List of processed token chunks
    """
    # Load prompts
    system_prompt, user_prompt_template = get_prompt_processing(
        prompt_path=process_prompt_path,
        few_shot_examples=few_shot_examples
    )
    
    # Initialize tracking
    processed_chunks = []
    history = ProcessingHistory()
    
    print(f"   ✓ Processing {len(token_chunks)} chunks with LLM...")
    print(f"   ✓ Using {len(few_shot_examples) if few_shot_examples else 0} few-shot examples")
    if allowed_labels:
        print(f"   ✓ Label scheme validation enabled with {len(allowed_labels)} allowed labels")
    
    # Process each chunk
    for idx, chunk in enumerate(tqdm(token_chunks, desc="Processing chunks")):

        processed_tokens, status, error_details = process_single_chunk(
            model=model,
            chunk=chunk,
            system_prompt=system_prompt,
            user_prompt_template=user_prompt_template,
            label_config=label_config,
            allowed_labels=allowed_labels,
        )
    
        processed_chunks.append(processed_tokens)
        history.add(status, idx, decode(processed_tokens), error_details)
        
        if status != "Success" and not status.startswith("Success (after"):
            print(f"   ⚠ Chunk {idx} failed: {status}")
    
    # Save history and results
    history.save(output_dir, filename)
    
    # Print summary
    summary = history.summary()
    print(f"\n   ✓ Processing completed:")
    print(f"      - Total chunks: {summary['total']}")
    print(f"      - Successful: {summary['success']}")
    print(f"      - Failed: {summary['total'] - summary['success']}")
    
    if output_dir and filename:
        json_path = os.path.join(output_dir, f"processed_chunks_{filename}.json")
        try:
            # Convert token lists to strings for JSON serialization
            chunks_as_strings = [decode(chunk) for chunk in processed_chunks]
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(chunks_as_strings, f, indent=4, ensure_ascii=False)
            print(f"   ✓ Processed chunks saved to: {json_path}")
        except Exception as e:
            print(f"   ✗ Error saving processed chunks: {e}")
    
    return processed_chunks

In [65]:
# ---------- Process chunks ----------

prompt_path = fr"{project_root}\llm_based_annotation\utils_extraction\prompts\simplified_parent_extraction_cot v2.txt"

processed_chunks = process_chunks(
    model=model,
    token_chunks=token_chunks,
    process_prompt_path=prompt_path,
    label_config=label_config,
    few_shot_examples=selected_few_shot_examples,
    output_dir=output_dir,
    filename=filename
)


   ✓ Processing 18 chunks with LLM...
   ✓ Using 30 few-shot examples


Processing chunks:   6%|▌         | 1/18 [01:13<20:53, 73.72s/it]

[' ', 'Ottawa', ',', ' ', 'Tuesday', ',', '\n', 'December', ' ', '12', ',', ' ', '1989', ' ', 'Appeal', '\n', 'No', '.', ' ', '2845', ' ', 'IN', ' ', 'THE', ' ', 'MATTER', ' ', 'OF', ' ', 'an', ' ', 'application', ' ', 'heard', '\n', 'May', '\xa0', '19', ',', ' ', '1989', ',', ' ', 'pursuant', ' ', 'to', ' ', 'section', ' ', '51', '.', '19', ' ', 'of', ' ', 'the', ' ', 'Excise', ' ', 'Tax', ' ', 'Act', ',', '\n', 'R', '.', 'S', '.', 'C', '.', ' ', '1970', ',', ' ', 'c', '.', ' ', 'E', '-', '13', ';', ' ', 'AND', ' ', 'IN', ' ', 'THE', ' ', 'MATTER', ' ', 'OF', ' ', 'a', ' ', 'decision', ' ', 'of', ' ', 'the', '\n', 'Minister', ' ', 'of', ' ', 'National', ' ', 'Revenue', ' ', 'dated', ' ', 'July', ' ', '10', ',', ' ', '1987', ',', ' ', 'with', ' ', 'respect', ' ', 'to', ' ', 'a', ' ', 'notice', ' ', 'of', '\n', 'objection', ' ', 'filed', ' ', 'pursuant', ' ', 'to', ' ', 'section', ' ', '51', '.', '17', ' ', 'of', ' ', 'the', ' ', 'Excise', ' ', 'Tax', ' ', 'Act', '.', ' ', 'BETWEEN', ' 

Processing chunks:  11%|█         | 2/18 [01:57<14:56, 56.04s/it]

[' ', 'The', ' ', 'sign', ' ', 'bridge', '\n', 'assemblies', ' ', 'do', ' ', 'not', ' ', 'afford', ' ', 'passage', ' ', 'nor', ' ', 'conveyance', '.', ' ', 'Place', ' ', 'of', ' ', 'Hearing', ':', ' ', 'Calgary', ',', '\n', 'Alberta', '\n', 'Date', ' ', 'of', ' ', 'Hearing', ':', ' ', 'May', ' ', '19', ',', ' ', '1989', '\n', 'Date', ' ', 'of', ' ', 'Decision', ':', ' ', 'December', '\n', '12', ',', ' ', '1989', ' ', 'Panel', ' ', 'Members', ':', ' ', 'Sidney', ' ', 'A', '.', ' ', 'Fraleigh', ',', '\n', 'Presiding', ' ', 'Member', ' ', 'Robert', '\n', 'J', '.', ' ', 'Bertrand', ',', ' ', 'Q', '.', 'C', '.', ',', ' ', 'Member', ' ', 'W', '.', '\n', 'Roy', ' ', 'Hines', ',', ' ', 'Member', ' ', 'Counsel', ' ', 'for', ' ', 'the', ' ', 'Tribunal', ':', ' ', 'Clifford', ' ', 'Sosnow', '\n', 'Clerk', ' ', 'of', ' ', 'the', ' ', 'Tribunal', ':', ' ', 'Lillian', '\n', 'Pharand', ' ', 'Appearances', ':', ' ', 'Douglas', '\n', 'Densmore', ',', ' ', 'for', ' ', 'the', ' ', 'appellant', ' ', 'Pete

...

[' ', 'The', '\n', 'documents', ' ', 'and', ' ', 'expert', ' ', 'evidence', ' ', 'apply', ' ', 'a', ' ', 'technical', ' ', 'and', ' ', 'scientific', ' ', 'standard', ' ', 'to', '\n', 'define', ' ', 'a', ' ', 'common', ' ', 'and', ' ', 'ordinary', ' ', 'word', '.', ' ', 'However', ',', '\n', 'the', ' ', 'Tribunal', ' ', 'considers', ' ', 'the', ' ', 'various', ' ', 'dictionary', ' ', 'definitions', ' ', 'submitted', ' ', 'by', ' ', 'the', '\n', 'parties', ' ', 'helpful', ' ', 'in', ' ', 'determining', ' ', 'the', ' ', 'common', ' ', 'meaning', ' ', 'of', ' ', 'the', ' ', 'word', '\n', '"', 'bridge', '"', ' ', 'in', ' ', 'the', ' ', 'exemption', ' ', 'clause', '.', ' ', 'In', ' ', 'the', ' ', 'Tribunal', "'", 's', ' ', 'view', ',', ' ', 'Driedger', ',', '\n', 'in', ' ', 'his', ' ', 'text', ' ', 'entitled', ' ', 'Construction', ' ', 'of', ' ', 'Statutes', ',', ' ', 'has', ' ', 'succinctly', ' ', 'commented', '\n', 'on', ' ', 'the', ' ', 'usefulness', ' ', 'of', ' ', 'dictionaries', ' ', '

Processing chunks:  94%|█████████▍| 17/18 [13:47<00:45, 45.98s/it]

[' ', 'The', ' ', 'goods', '\n', 'in', ' ', 'issue', ' ', 'are', ' ', 'built', ' ', 'solely', ' ', 'for', ' ', 'the', ' ', 'purpose', ' ', 'of', ' ', 'supporting', ' ', 'signs', ' ', 'and', '/', 'or', ' ', 'lights', ' ', 'to', '\n', 'inform', ' ', 'drivers', ' ', 'and', ' ', 'direct', ' ', 'vehicular', ' ', 'traffic', '.', ' ', 'However', ',', '\n', 'the', ' ', 'appellant', ' ', 'has', ' ', 'also', ' ', 'argued', ' ', 'that', ' ', 'the', ' ', 'tertiary', ' ', 'definition', ' ', 'of', ' ', 'the', ' ', 'word', '\n', '"', 'bridge', ':', '"', ' ', '"', 'something', ' ', 'resembling', ' ', 'a', ' ', 'bridge', ' ', '(', 'as', ' ', 'in', ' ', 'serving', ' ', 'as', ' ', 'a', '\n', 'support', ' ', 'for', ' ', 'or', ' ', 'a', ' ', 'way', ' ', 'over', ' ', 'something', ' ', 'else', ')', ' ', 'as', ' ', '.', '.', '.', ' ', 'a', ' ', 'framework', ' ', 'that', ' ', 'spans', '\n', 'railroad', ' ', 'tracks', ' ', 'and', ' ', 'supports', ' ', 'signals', '"', ' ', 'provides', ' ', 'support', ' ', 'for',

Processing chunks: 100%|██████████| 18/18 [14:14<00:00, 47.45s/it]

[' ', '2', '.', ' ', 'S', '.', 'C', '.', ' ', '1988', ',', ' ', 'c', '.', ' ', '56', '.', ' ', '[', '3', ']', '.', ' ', '[', '1977', ']', ' ', '1', ' ', 'S', '.', 'C', '.', 'R', '.', ' ', '456', '.', ' ', '[', '4', ']', '.', ' ', '2', ' ', 'T', '.', 'B', '.', 'R', '.', ' ', '106', '.', ' ', '[', '5', ']', '.', ' ', 'Driedger', ',', ' ', 'E', '.', ' ', 'A', '.', ',', ' ', 'Construction', '\n', 'of', ' ', 'Statutes', ' ', '(', '2nd', ' ', 'ed', '.', ')', '.', ' ', '[', '6', ']', '.', ' ', 'Wilson', ' ', 'v', '.', ' ', 'Town', ' ', 'of', ' ', 'Barnstead', '\n', '(', '1906', ')', ',', ' ', '74', ' ', 'N', '.', 'H', '.', ' ', '78', '.', ' ', '[', '7', ']', '.', ' ', 'City', ' ', 'of', ' ', 'Chicago', ' ', 'v', '.', '\n', 'Pittsburgh', ',', ' ', 'Ft', '.', ' ', 'W', '.', ' ', '&', 'amp', ';', ' ', 'C', '.', ' ', 'R', '.', ' ', 'Co', '.', ',', ' ', '(', '1910', ')', ' ', '93', ' ', 'N', '.', 'E', '.', ' ', '307', ',', ' ', '308', '.', ' ', '[', '8', ']', '.', ' ', 'Proprietors', ' ', 'of', ' 

In [53]:
def verif_deri(ori, deri):
    deri2 = []
    for el in deri:
        if not el.startswith("<"):
            deri2.append(el)


    #deri2[0] = " "
    print(deri2)

    print(ori == deri2)

In [ ]:
ori = ['\n', '[', 'Italics', ' ', 'in', ' ', 'the', ' ', 'original', ';', ' ', 'boldface', ' ', 'added', ']', '\n', '*', ' ', '*', ' ', '*', '\n', '[', '7', ']', ' ', 'I', ' ', 'now', ' ', 'consider', ' ', 'whether', ' ', 'the', ' ', 'applicants', ' ', 'have', ' ', 'shown', '\n', 'that', ' ', 'the', ' ', 'filing', ' ', 'of', ' ', 'the', ' ', 'appeals', ' ', 'is', ' ', 'likely', ' ', 'to', ' ', 'cause', ' ', 'serious', '\n', 'or', ' ', 'irreparable', ' ', 'prejudice', ' ', 'to', ' ', 'the', ' ', 'holders', ' ', 'of', ' ', 'the', ' ', 'section', ' ', '23', ' ', 'rights', ' ', 'vindicated', ' ', 'in', '\n', 'the', ' ', 'impugned', ' ', 'judgment', '.', '\n', '[', '8', ']', ' ', 'It', ' ', 'is', ' ', 'important', ' ', 'to', ' ', 'begin', ' ', 'by', ' ', 'clarifying', ' ', 'the', '\n', 'rights', ' ', 'at', ' ', 'issue', '.', ' ', 'To', ' ', 'do', ' ', 'so', ',', ' ', 'it', ' ', 'is', ' ', 'necessary', ' ', 'to', ' ', 'briefly', ' ', 'discuss', ' ', 'section', ' ', '23', ' ', 'of', ' ', 'the', '\n', 'Charter', ' ', 'as', ' ', 'well', ' ', 'as', ' ', 'the', ' ', 'relevant', ' ', 'aspects', ' ', 'of', ' ', 'Blanchard', ' ', 'J', '.', '’', 's', ' ', 'judgment', '.', '\n', '[', '9', ']', ' ', 'Section', ' ', '23', ' ', 'grants', ' ', 'a', ' ', 'right', ' ', 'to', ' ', 'minority', '-', 'language', ' ', 'instruction', '\n', 'to', ' ', 'certain', ' ', 'parents', ' ', 'who', ' ', 'form', ' ', 'part', ' ', 'of', ' ', 'the', ' ', 'French', ' ', 'or', ' ', 'English', ' ', 'linguistic', ' ', 'minority', '\n', 'population', ' ', 'of', ' ', 'the', ' ', 'province', ' ', 'in', ' ', 'which', ' ', 'they', ' ', 'reside', '.', ' ', 'The', '\n', 'scope', ' ', 'of', ' ', 'that', ' ', 'right', ',', ' ', 'which', ' ', 'depends', ' ', 'on', ' ', 'the', ' ', 'number', ' ', 'of', ' ', 'children', ' ', 'whose', ' ', 'parents', ' ', 'hold', '\n', 'section', ' ', '23', ' ', 'rights', ',', ' ', 'varies', ' ', 'from', ' ', 'a', ' ', 'right', ' ', 'to', '\n', 'publicly', '-', 'funded', ' ', 'minority', '-', 'language', ' ', 'instruction', ' ', '(', 'section', ' ', '23', '(', '3', ')', '(', 'a', ')', ')', ' ', 'to', ' ', 'a', ' ', 'right', ' ', 'to', ' ', 'receive', ' ', 'such', ' ', 'instruction', ' ', 'in', ' ', 'publicly', '-', 'funded', ' ', 'minority', '-', 'language', '\n', 'educational', ' ', 'facilities', ' ', '(', 'section', ' ', '23', '(', '3', ')', '(', 'b', ')', ')', '.', ' ', 'The', ' ', 'latter', ' ', 'right', ',', ' ', 'which', ' ', 'is', ' ', 'situated', '\n', 'at', ' ', 'the', ' ', 'high', ' ', 'end', ' ', 'of', ' ', 'what', ' ', 'has', ' ', 'been', ' ', 'described', ' ', 'as', ' ', 'a', ' ', '“', 'sliding', ' ', 'scale', '”', ',', '[', '17', ']', ' ', 'comprises', ' ', 'a', ' ', 'right', ' ', 'of', ' ', 'management', ' ', 'and', ' ', 'control', '\n', 'that', ' ', 'includes', ' ', 'authority', ' ', 'to', ' ', 'make', ' ', 'decisions', ' ', 'relating', ' ', 'to', ' ', 'the', ' ', 'recruitment', ' ', 'and', '\n', 'assignment', ' ', 'of', ' ', 'teachers', ' ', 'and', ' ', 'other', ' ', 'personnel', '.', '[', '18', ']', '\n', '[', '10', ']', ' ', 'The', ' ', 'applicants', ' ', 'represent', ' ', 'the', ' ', 'interests', ' ', 'of', ' ', 'parents', '\n', 'who', ' ', 'form', ' ', 'part', ' ', 'of', ' ', 'Quebec', '’', 's', ' ', 'English', ' ', 'minority', ' ', 'population', ' ', 'and', ' ', 'who', ',', ' ', 'unquestionably', ',', '\n', 'have', ' ', 'a', ' ', 'right', ' ', 'to', ' ', 'publicly', '-', 'funded', ' ', 'minority', '-', 'language', '\n', 'educational', ' ', 'facilities', ',', ' ', 'as', ' ', 'well', ' ', 'as', ' ', 'a', ' ', 'right', ' ', 'to', ' ', 'manage', ' ', 'and', ' ', 'control', ' ', 'such', '\n', 'facilities', '.']
deri = ['<start>', '\n', '[', 'Italics', ' ', 'in', ' ', 'the', ' ', 'original', ';', ' ', 'boldface', ' ', 'added', ']', '\n', '*', ' ', '*', ' ', '*', '\n', '[', '7', ']', ' ', 'I', ' ', 'now', ' ', 'consider', ' ', 'whether', ' ', 'the', ' ', 'applicants', ' ', 'have', ' ', 'shown', '\n', 'that', ' ', 'the', ' ', 'filing', ' ', 'of', ' ', 'the', ' ', 'appeals', ' ', 'is', ' ', 'likely', ' ', 'to', ' ', 'cause', ' ', 'serious', '\n', 'or', ' ', 'irreparable', ' ', 'prejudice', ' ', 'to', ' ', 'the', ' ', 'holders', ' ', 'of', ' ', 'the', ' ', '<legislation>', 'section', ' ', '23', '</legislation>', ' ', 'rights', ' ', 'vindicated', ' ', 'in', '\n', 'the', ' ', 'impugned', ' ', 'judgment', '.', '\n', '[', '8', ']', ' ', 'It', ' ', 'is', ' ', 'important', ' ', 'to', ' ', 'begin', ' ', 'by', ' ', 'clarifying', ' ', 'the', '\n', 'rights', ' ', 'at', ' ', 'issue', '.', ' ', 'To', ' ', 'do', ' ', 'so', ',', ' ', 'it', ' ', 'is', ' ', 'necessary', ' ', 'to', ' ', 'briefly', ' ', 'discuss', ' ', '<legislation>', 'section', ' ', '23', ' ', 'of', ' ', 'the', '\n', 'Charter', '</legislation>', ' ', 'as', ' ', 'well', ' ', 'as', ' ', 'the', ' ', 'relevant', ' ', 'aspects', ' ', 'of', ' ', '<decision>', 'Blanchard', ' ', 'J', '.', '’', 's', ' ', 'judgment', '</decision>', '.', '\n', '[', '9', ']', ' ', '<legislation>', 'Section', ' ', '23', '</legislation>', ' ', 'grants', ' ', 'a', ' ', 'right', ' ', 'to', ' ', 'minority', '-', 'language', ' ', 'instruction', '\n', 'to', ' ', 'certain', ' ', 'parents', ' ', 'who', ' ', 'form', ' ', 'part', ' ', 'of', ' ', 'the', ' ', 'French', ' ', 'or', ' ', 'English', ' ', 'linguistic', ' ', 'minority', '\n', 'population', ' ', 'of', ' ', 'the', ' ', 'province', ' ', 'in', ' ', 'which', ' ', 'they', ' ', 'reside', '.', ' ', 'The', '\n', 'scope', ' ', 'of', ' ', 'that', ' ', 'right', ',', ' ', 'which', ' ', 'depends', ' ', 'on', ' ', 'the', ' ', 'number', ' ', 'of', ' ', 'children', ' ', 'whose', ' ', 'parents', ' ', 'hold', '\n', '<legislation>', 'section', ' ', '23', '</legislation>', ' ', 'rights', ',', ' ', 'varies', ' ', 'from', ' ', 'a', ' ', 'right', ' ', 'to', '\n', 'publicly', '-', 'funded', ' ', 'minority', '-', 'language', ' ', 'instruction', ' ', '(', '<legislation>', 'section', ' ', '23', '(', '3', ')', '(', 'a', ')', '</legislation>', ')', ' ', 'to', ' ', 'a', ' ', 'right', ' ', 'to', ' ', 'receive', ' ', 'such', ' ', 'instruction', ' ', 'in', ' ', 'publicly', '-', 'funded', ' ', 'minority', '-', 'language', '\n', 'educational', ' ', 'facilities', ' ', '(', '<legislation>', 'section', ' ', '23', '(', '3', ')', '(', 'b', ')', '</legislation>', ')', '.', ' ', 'The', ' ', 'latter', ' ', 'right', ',', ' ', 'which', ' ', 'is', ' ', 'situated', '\n', 'at', ' ', 'the', ' ', 'high', ' ', 'end', ' ', 'of', ' ', 'what', ' ', 'has', ' ', 'been', ' ', 'described', ' ', 'as', ' ', 'a', ' ', '“', 'sliding', ' ', 'scale', '”', ',', '[', '17', ']', ' ', 'comprises', ' ', 'a', ' ', 'right', ' ', 'of', ' ', 'management', ' ', 'and', ' ', 'control', '\n', 'that', ' ', 'includes', ' ', 'authority', ' ', 'to', ' ', 'make', ' ', 'decisions', ' ', 'relating', ' ', 'to', ' ', 'the', ' ', 'recruitment', ' ', 'and', '\n', 'assignment', ' ', 'of', ' ', 'teachers', ' ', 'and', ' ', 'other', ' ', 'personnel', '.', '[', '18', ']', '\n', '[', '10', ']', ' ', 'The', ' ', 'applicants', ' ', 'represent', ' ', 'the', ' ', 'interests', ' ', 'of', ' ', 'parents', '\n', 'who', ' ', 'form', ' ', 'part', ' ', 'of', ' ', 'Quebec', '’', 's', ' ', 'English', ' ', 'minority', ' ', 'population', ' ', 'and', ' ', 'who', ',', ' ', 'unquestionably', ',', '\n', 'have', ' ', 'a', ' ', 'right', ' ', 'to', ' ', 'publicly', '-', 'funded', ' ', 'minority', '-', 'language', '\n', 'educational', ' ', 'facilities', ',', ' ', 'as', ' ', 'well', ' ', 'as', ' ', 'a', ' ', 'right', ' ', 'to', ' ', 'manage', ' ', 'and', ' ', 'control', ' ', 'such', '\n', 'facilities', '.', '\n', '<end>']


verif_deri(ori, deri)

['\n', 'The', ' ', 'appellant', ' ', 'argued', '\n', 'that', ' ', 'the', ' ', 'goods', ' ', 'in', ' ', 'issue', ' ', 'fell', ' ', 'within', ' ', 'the', ' ', 'ordinary', ' ', 'and', ' ', 'common', ' ', 'meaning', ' ', 'of', ' ', 'the', ' ', 'word', '\n', '"', 'bridge', '"', ' ', 'for', ' ', 'three', ' ', 'reasons', '.', ' ', 'Initially', ',', ' ', 'the', ' ', 'goods', ' ', 'in', ' ', 'issue', ' ', 'span', '\n', 'across', ' ', 'a', ' ', 'highway', ',', ' ', 'have', ' ', 'supports', ' ', 'at', ' ', 'either', ' ', 'end', ' ', 'of', ' ', 'the', ' ', 'structure', ' ', 'and', ' ', 'carry', ' ', 'a', ' ', 'load', '.', ' ', 'Secondly', ',', '\n', 'goods', ' ', 'like', ' ', 'those', ' ', 'in', ' ', 'issue', ' ', 'are', ' ', 'called', ' ', '"', 'sign', ' ', 'bridges', '"', ' ', 'by', ' ', 'the', ' ', 'people', ' ', 'who', '\n', 'designed', ' ', 'these', ' ', 'structures', ',', ' ', 'the', ' ', 'suppliers', ' ', 'of', ' ', 'components', ' ', 'used', ' ', 'to', ' ', 'build', ' ', 'the', '\n', 'stru

In [20]:
#write the processed chuncks in a json file for later use in the annotation interface

import json 
with open(f"{output_dir}\\processed_chunks.json", "w") as f:
    json.dump(processed_chunks, f)

## Post Processing

In [37]:
# Read the processed_chuncks.json file to verify it was written correctly
import json
with open(f"{output_dir}\\processed_chunks.json", "r") as f:
    processed_chunks = json.load(f)

In [21]:
# Processed_chunks is a list of lists of tokens, we need to flatten it to get a single list of tokens for the whole document
processed_tokens_flat = flatten_token_chunks(processed_chunks)


# Read in parallel the original tokens and the processed tokens. Always prefer the original tokens, but if there is an auto_label token in the processed tokens, 
# we want to keep it and merge it with the original tokens. 
# This way we can keep the original formatting and structure of the document while adding the auto_labels extracted by the model.
original_tokens = tokenize(html_content)
processed_html_content_tokens = merge_tokens_with_auto_labels(original_tokens, processed_tokens_flat)

# This merging process can sometimes create some formatting issues with brackets, we need to correct them to get a valid HTML structure.
processed_html_content_tokens_corrected = correct_tokens_brackets(processed_html_content_tokens)
assert check_tokens_brackets(processed_html_content_tokens_corrected), "The brackets in the merged tokens are not balanced. Please check the merging and bracket correction steps for errors."



# The correction of the brackets can sometimes create some redoundant or useless formatting  with the HTML, we need to clean it to compare it with the original.
processed_html = decode(processed_html_content_tokens_corrected)
processed_html_cleaned = clean_html_formatting(processed_html)
print(f"\nMerged HTML length: {len(processed_html_cleaned)}")


# This step is just to ensure a good visualisation of HTMLLabelizer and to add the necessary attribute to stay consistent with the label scheme
processed_html_content = add_attributes_to_auto_labels(processed_html_cleaned)


# Last check of the final processed_html_content with the original HTML, ignoring the auto_label tags which are not present in the original HTML but only in the processed one.
comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)
assert comparison_result, "The processed HTML content does not match the original HTML content when ignoring auto_label tags. Please check the merging and post-processing steps for errors."


# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_llm_{version}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")

   ✓ Flattened 18 chunks into 9763 tokens

Merged HTML length: 115569
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)
   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1989CanLII1415ONCA\v_prompt_2_500_100_30_gpt5.2_chunk2_subdef2
